# Testing classifiers and models

In [1]:
# Cell 1
#=================== SETUP ===================

import os
import glob
import random
import numpy as np
import tensorflow as tf
import mimetypes

# ── Disable XLA/JIT compilation ──────────────────────────────────────────────
# Prevents "JIT compilation failed [Rsqrt]" UnknownError on BatchNormalization.
# Root cause: libdevice (CUDA PTX library) not found in standard paths on this server.
# Disabling JIT has minimal performance impact (~5%) and fixes the crash completely.
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
# Tell XLA exactly where CUDA's libdevice lives on this server.
# Fixes: "libdevice not found at ./libdevice.10.bc" InternalError during model.fit().
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/nvidia-cuda-toolkit'
tf.config.optimizer.set_jit(False)
# ─────────────────────────────────────────────────────────────────────────────

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU memory growth enabled")
    except RuntimeError as e:
        print(e)

tf.keras.backend.clear_session()
print("✅ Setup complete")

2026-04-21 10:14:53.350787: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✅ GPU memory growth enabled
✅ Setup complete


In [2]:
# Cell 2
# =================== PATHS & DATA SOURCES ===================

import os

GDRIVE_PATH       = os.path.expanduser('~/RealEyes/gdrive')
BASE_PROJECT_PATH = os.path.join(GDRIVE_PATH, 'deepfake_image_project', 'code')
CODE_PATH         = os.path.join(GDRIVE_PATH, 'code')
DATA_PATH         = os.path.join(GDRIVE_PATH, 'data_set')
DATA_SPLIT_PATH   = os.path.join(GDRIVE_PATH, 'data_set_split')
DATASET_ROOT      = DATA_SPLIT_PATH

# ── Single dedicated folder for ALL models from this notebook ────────────────
# On disk  : ~/RealEyes/gdrive/deepfake_image_project/models/RealEyes_v2/
# In GDrive: deepfake_image_project/models/RealEyes_v2/
# Sub-folders are created automatically as each model is saved.
MODELS_DIR = os.path.join(GDRIVE_PATH, 'deepfake_image_project', 'models', 'RealEyes_v2')

# ── Per-model sub-folders ─────────────────────────────────────────────────────
# Each model type gets its own folder so old and new runs are never mixed up.
# Google Drive will show:  deepfake_image_project/models/RealEyes_v2/
#   ├── autoencoder/   ← encoder + autoencoder files
#   ├── cnn_srm/       ← CNN-SRM checkpoints and final models
#   ├── efficientnet/  ← EfficientNetB0 stage-1 and stage-2
#   └── two_stream/    ← Two-Stream phase-1, phase-2, and final
AE_DIR      = os.path.join(MODELS_DIR, 'autoencoder')
CNN_SRM_DIR = os.path.join(MODELS_DIR, 'cnn_srm')
EFF_DIR     = os.path.join(MODELS_DIR, 'efficientnet')
TS_DIR      = os.path.join(MODELS_DIR, 'two_stream')
for _d in [AE_DIR, CNN_SRM_DIR, EFF_DIR, TS_DIR]:
    os.makedirs(_d, exist_ok=True)

# Backwards-compat alias used throughout the notebook
AE_SAVE_DIR = AE_DIR

# ── Run tagging + smart resume ────────────────────────────────────────────────
from datetime import datetime
RUN_TAG       = datetime.now().strftime('%Y%m%d_%H%M')
SKIP_IF_SAVED = False  # False = always retrain from scratch; True = load saved model

def find_latest_model(filename): #if we want to load a model we write here the name of the model
    """Return path to `filename` anywhere under MODELS_DIR, or None if missing."""
    candidates = glob.glob(os.path.join(MODELS_DIR, '**', filename), recursive=True)
    if not candidates:
        return None
    return max(candidates, key=os.path.getmtime)

def save_model_versioned(model, save_dir, base_name):
    """Save model as base_name.keras (latest) + base_name_vTAG.keras (versioned backup).
    save_dir should be one of AE_DIR, CNN_SRM_DIR, EFF_DIR, TS_DIR."""
    os.makedirs(save_dir, exist_ok=True)
    latest    = os.path.join(save_dir, f'{base_name}.keras')
    versioned = os.path.join(save_dir, f'{base_name}_v{RUN_TAG}.keras')
    model.save(latest)
    model.save(versioned)
    print(f"  ✅ Saved  → {latest}")
    print(f"  📦 Backup → {versioned}")
    push_models_to_gdrive()
    return latest
# ─────────────────────────────────────────────────────────────────────────────

# Helper: push a local folder to Google Drive via rclone
def push_models_to_gdrive():
    import subprocess
    remote = "gdrive:deepfake_image_project/models/RealEyes_v2"
    result = subprocess.run(
        ["rclone", "sync", MODELS_DIR, remote, "--transfers=4"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ☁️  Models pushed to Google Drive → {remote}")
    else:
        print(f"  ⚠️  rclone push failed: {result.stderr[:200]}")

# Permanent datasets folder (notebook working directory)
# CiFake and OpenForensics were copied here from /tmp (permanent across reboots)
DATASETS_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                             os.path.expanduser('~/RealEyes/RealEyes/datasets'))
DATASETS_DIR = os.path.expanduser('~/RealEyes/RealEyes/datasets')

CELEBDF_DIR = os.path.join(DATASETS_DIR, 'celebdf_v2')

# ── helper: only add a dataset if its folder actually exists ──────────────────
def _add_if_exists(d, name, path):
    if os.path.isdir(path):
        d[name] = path
    else:
        print(f"  ⚠️  {name} not found at {path} — skipping")

# ── CiFake toggle ────────────────────────────────────────────────────────────
# CiFake = 100 k low-res (32×32 upscaled) AI-generated images, NOT face deepfakes.
# ✅ Set USE_CIFAKE = True  → adds ~100 k extra training samples (may hurt precision on real fakes)
# ✅ Set USE_CIFAKE = False → cleaner domain match; recommended if val accuracy drops
USE_CIFAKE = True
# ─────────────────────────────────────────────────────────────────────────────

# ---------------- TRAIN ----------------
train_datasets = {}
_add_if_exists(train_datasets, 'OpenForensics',
               os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Train'))
_add_if_exists(train_datasets, 'CustomWar',
               os.path.join(DATASET_ROOT, 'train'))
if USE_CIFAKE:
    _add_if_exists(train_datasets, 'CiFake',
                   os.path.join(DATASETS_DIR, 'cifake/train'))

# ---------------- VALIDATION ----------------
validation_datasets = {}
_add_if_exists(validation_datasets, 'OpenForensics',
               os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Validation'))
_add_if_exists(validation_datasets, 'CustomWar',
               os.path.join(DATASET_ROOT, 'val'))
if USE_CIFAKE:
    _add_if_exists(validation_datasets, 'CiFake',
                   os.path.join(DATASETS_DIR, 'cifake/test'))

# ---------------- TEST ----------------
# CiFake has no independent test split (test == val), so always omitted from test.
test_datasets = {}
_add_if_exists(test_datasets, 'OpenForensics',
               os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Test'))
_add_if_exists(test_datasets, 'CustomWar',
               os.path.join(DATASET_ROOT, 'test'))

# Celeb-DF v2 — auto-detected if frames have been extracted
for split, d in [('train', train_datasets), ('val', validation_datasets), ('test', test_datasets)]:
    _add_if_exists(d, 'CelebDF', os.path.join(CELEBDF_DIR, split))

print("✅ Paths ready")
print(f"  MODELS_DIR  : {MODELS_DIR}")
print(f"  DATASETS_DIR: {DATASETS_DIR}")
print(f"  RUN_TAG     : {RUN_TAG}  (SKIP_IF_SAVED={SKIP_IF_SAVED})")
print("\n📋 Active datasets:")
for split, d in [('TRAIN', train_datasets), ('VAL', validation_datasets), ('TEST', test_datasets)]:
    print(f"  {split}: {list(d.keys())}")

✅ Paths ready
  MODELS_DIR  : /home/sceuser/RealEyes/gdrive/deepfake_image_project/models/RealEyes_v2
  DATASETS_DIR: /home/sceuser/RealEyes/RealEyes/datasets
  RUN_TAG     : 20260421_1015  (SKIP_IF_SAVED=False)

📋 Active datasets:
  TRAIN: ['OpenForensics', 'CustomWar', 'CiFake', 'CelebDF']
  VAL: ['OpenForensics', 'CustomWar', 'CiFake', 'CelebDF']
  TEST: ['OpenForensics', 'CustomWar', 'CelebDF']


In [3]:
# Cell 3
# =================== SYNC GOOGLE DRIVE → LOCAL (rclone) ===================
# Syncs raw images, saved models and CelebDF videos FROM Google Drive TO the server.
# Safe to re-run: rclone skips files already identical (size + mod-time check).

import subprocess, os

RCLONE_REMOTE     = "gdrive:deepfake_image_project"   # root of your GDrive project folder
LOCAL_GDRIVE_ROOT = os.path.expanduser("~/RealEyes/gdrive")

# Each entry: (remote sub-path, absolute local destination)
# remote_sub is relative to RCLONE_REMOTE (gdrive:deepfake_image_project/)
SYNC_TARGETS = [
    ("data_set",
     os.path.join(LOCAL_GDRIVE_ROOT, "data_set")),

    ("models",
     os.path.join(LOCAL_GDRIVE_ROOT, "deepfake_image_project", "models")),

    ("datasets/celebdf_v2_videos",
     os.path.join(LOCAL_GDRIVE_ROOT, "deepfake_image_project", "datasets", "celebdf_v2_videos")),
]

print("☁️  Syncing Google Drive → local server...")
print("   (Fast after first run — rclone skips unchanged files)\n")

for remote_sub, local_path in SYNC_TARGETS:
    remote_path = f"{RCLONE_REMOTE}/{remote_sub}"
    os.makedirs(local_path, exist_ok=True)
    print(f"  ⬇️  {remote_path}")
    print(f"       → {local_path}")
    result = subprocess.run(
        ["rclone", "sync", remote_path, local_path, "--transfers=8"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  ⚠️  rclone error: {result.stderr.strip()[:200]}")
    else:
        print(f"  ✅ Done")

for cls in ["REAL", "FAKE"]:
    path = os.path.join(LOCAL_GDRIVE_ROOT, "data_set", cls)
    count = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]) if os.path.isdir(path) else 0
    print(f"  data_set/{cls}: {count:,} images")

print("\n✅ Sync complete")

☁️  Syncing Google Drive → local server...
   (Fast after first run — rclone skips unchanged files)

  ⬇️  gdrive:deepfake_image_project/data_set
       → /home/sceuser/RealEyes/gdrive/data_set
  ✅ Done
  ⬇️  gdrive:deepfake_image_project/models
       → /home/sceuser/RealEyes/gdrive/deepfake_image_project/models
  ✅ Done
  ⬇️  gdrive:deepfake_image_project/datasets/celebdf_v2_videos
       → /home/sceuser/RealEyes/gdrive/deepfake_image_project/datasets/celebdf_v2_videos
  ✅ Done
  data_set/REAL: 4,190 images
  data_set/FAKE: 4,190 images

✅ Sync complete


## Celeb-DF v2 — Setup (one-time)

**Why:** Celeb-DF v2 contains high-quality celebrity deepfakes that are much harder to detect than CiFake. Adding it helps the model learn real distinguishing features.

**How to download:**
1. Go to: https://github.com/yuezunli/celeb-deepfakeforensics  
2. Fill the short Google Form to get the Google Drive download links (usually approved in minutes)  
3. Download the ZIP files and upload them to your Google Drive under:  
   `deepfake_image_project/datasets/celebdf_v2_videos/`  
   - `Celeb-real.zip` (real celebrity videos)  
   - `Celeb-synthesis.zip` (synthesized/fake videos)  
   - `List_of_testing_videos.txt` (official test/train split list)

**After uploading**, run the two cells below — they will:  
1. Extract frames from every video (1 frame per second by default)  
2. Split them into `train / val / test` following the official test list  
3. Save frames to `./datasets/celebdf_v2/`  

If you have not downloaded Celeb-DF v2 yet, skip these cells — the code will detect its absence and train without it.

In [ ]:
# Cell 5
#==================DO NOT RUN THIS CELL==================
# =================== CELEB-DF v2: FRAME EXTRACTION (one-time setup) ===================
# Your Google Drive has the videos already extracted (not zipped) at:
#   deepfake_image_project/datasets/celebdf_v2_videos/
#     ├── Celeb-real/          ← real celebrity videos (.mp4)
#     ├── Celeb-synthesis/     ← deepfake videos (.mp4)
#     ├── YouTube-real/        ← real YouTube videos (.mp4)
#     └── List_of_testing_videos.txt
#
# Cell 3 (rclone sync) downloads this folder to the server.
# This cell reads those videos and saves 1 frame/sec as .jpg images.
# Safe to re-run: already-extracted frames are skipped.

import os, cv2, random, shutil

# Source: locally synced from Google Drive by Cell 3
CELEBDF_VIDEOS_DIR = os.path.join(GDRIVE_PATH, "deepfake_image_project", "datasets", "celebdf_v2_videos")
CELEBDF_OUT_DIR    = os.path.expanduser("~/RealEyes/RealEyes/datasets/celebdf_v2")
CELEBDF_TEST_LIST  = os.path.join(CELEBDF_VIDEOS_DIR, "List_of_testing_videos.txt")
FRAMES_PER_SEC     = 1    # extract 1 frame per second (increase for more images)
MAX_FRAMES_VIDEO   = 30   # cap per video to avoid class imbalance

def extract_frames(video_path, out_dir, fps=1, max_frames=30):
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 25
    interval  = max(1, int(video_fps / fps))
    count, saved = 0, 0
    while cap.isOpened() and saved < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if count % interval == 0:
            fname = os.path.join(out_dir, f"frame_{saved:04d}.jpg")
            if not os.path.exists(fname):
                cv2.imwrite(fname, frame)
            saved += 1
        count += 1
    cap.release()
    return saved

def setup_celebdf(videos_dir, out_dir, test_list_path, fps=1, max_frames=30, seed=42):
    random.seed(seed)

    if not os.path.isdir(videos_dir):
        print(f"⚠️  {videos_dir} not found — run Cell 3 first to sync from Google Drive")
        return False

    # Load official test video names
    test_videos = set()
    if os.path.exists(test_list_path):
        with open(test_list_path) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    test_videos.add(os.path.splitext(os.path.basename(parts[-1]))[0])
    print(f"Official test videos: {len(test_videos)}")

    sources = {
        "REAL": ["Celeb-real", "YouTube-real"],
        "FAKE": ["Celeb-synthesis"],
    }

    for label, subdirs in sources.items():
        video_files = []
        for sub in subdirs:
            vid_dir = os.path.join(videos_dir, sub)
            if os.path.isdir(vid_dir):
                video_files += [(os.path.join(vid_dir, f), f)
                                for f in os.listdir(vid_dir)
                                if f.lower().endswith(('.mp4', '.avi', '.mov'))]

        random.shuffle(video_files)
        n = len(video_files)
        n_test = len([f for _, f in video_files
                      if os.path.splitext(f)[0] in test_videos]) or max(1, int(n * 0.10))

        test_set  = [v for v in video_files if os.path.splitext(v[1])[0] in test_videos]
        rest      = [v for v in video_files if v not in test_set]
        n_val     = max(1, int(len(rest) * 0.10))
        val_set   = rest[:n_val]
        train_set = rest[n_val:]

        for split_name, videos in [("train", train_set), ("val", val_set), ("test", test_set)]:
            split_dir = os.path.join(out_dir, split_name, label)
            os.makedirs(split_dir, exist_ok=True)
            for vid_path, vid_name in videos:
                vname = os.path.splitext(vid_name)[0]
                frame_dir = os.path.join(split_dir, vname)
                saved = extract_frames(vid_path, frame_dir, fps, max_frames)
            print(f"  {split_name:5s}/{label}: {len(videos)} videos processed")

    print(f"\n✅ Celeb-DF v2 frames saved to {out_dir}")
    return True

celebdf_available = setup_celebdf(
    CELEBDF_VIDEOS_DIR, CELEBDF_OUT_DIR,
    CELEBDF_TEST_LIST, fps=FRAMES_PER_SEC, max_frames=MAX_FRAMES_VIDEO
)

Official test videos: 518


  train/REAL: 641 videos processed
  val  /REAL: 71 videos processed
  test /REAL: 178 videos processed
  train/FAKE: 4770 videos processed
  val  /FAKE: 529 videos processed
  test /FAKE: 340 videos processed

✅ Celeb-DF v2 frames saved to /home/sceuser/RealEyes/RealEyes/datasets/celebdf_v2


In [ ]:
# Cell 6
#==================DO NOT RUN THIS CELL==================
# =================== AUTO SPLIT: data_set → data_set_split ===================
# Splits CustomWar images (DATA_PATH) into train/val/test (80/10/10).
# Safe to re-run: only copies files that are not already in the destination.
# Run this cell ONCE whenever new images are added to data_set.

import os, shutil, random

SPLIT_RATIOS = (0.80, 0.10, 0.10)   # train / val / test
VALID_EXTS   = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}
SPLIT_SEED   = 42

def auto_split_dataset(src_root, dst_root, ratios=SPLIT_RATIOS, seed=SPLIT_SEED):
    random.seed(seed)
    splits = ["train", "val", "test"]
    classes = ["FAKE", "REAL"]

    total_copied = 0

    for cls in classes:
        src_cls = os.path.join(src_root, cls)
        if not os.path.isdir(src_cls):
            print(f"⚠️  {src_cls} not found — skipping")
            continue

        all_files = sorted([
            f for f in os.listdir(src_cls)
            if os.path.splitext(f)[1].lower() in VALID_EXTS
        ])
        random.shuffle(all_files)

        n = len(all_files)
        n_train = int(n * ratios[0])
        n_val   = int(n * ratios[1])
        buckets  = {
            "train": all_files[:n_train],
            "val":   all_files[n_train : n_train + n_val],
            "test":  all_files[n_train + n_val:]
        }

        for split, files in buckets.items():
            dst_cls = os.path.join(dst_root, split, cls)
            os.makedirs(dst_cls, exist_ok=True)

            copied = 0
            for fname in files:
                src_file = os.path.join(src_cls, fname)
                dst_file = os.path.join(dst_cls, fname)
                if not os.path.exists(dst_file):   # skip if already there
                    shutil.copy2(src_file, dst_file)
                    copied += 1
            total_copied += copied
            print(f"   {split:5s}/{cls:4s}: {len(files):4d} files  ({copied} newly copied)")

    print(f"\n✅ Split complete — {total_copied} new files copied to {dst_root}")

print(f"📂 Source : {DATA_PATH}")
print(f"📂 Dest   : {DATA_SPLIT_PATH}")
print()
auto_split_dataset(DATA_PATH, DATA_SPLIT_PATH)

# Verify totals
for split in ["train", "val", "test"]:
    total = sum(
        len(os.listdir(os.path.join(DATA_SPLIT_PATH, split, cls)))
        for cls in ["FAKE", "REAL"]
        if os.path.isdir(os.path.join(DATA_SPLIT_PATH, split, cls))
    )
    print(f"  {split:5s}: {total:,} images")

# ── Push split back to Google Drive so it's visible there ────────────────────
import subprocess
print("\n☁️  Pushing data_set_split → Google Drive...")
result = subprocess.run(
    ["rclone", "sync",
     DATA_SPLIT_PATH,
     "gdrive:deepfake_image_project/data_set_split",
     "--transfers=8", "--progress"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"  ⚠️  rclone push failed:\n{result.stderr[:300]}")
else:
    print("  ✅ data_set_split is now visible in Google Drive")

📂 Source : /home/sceuser/RealEyes/gdrive/data_set
📂 Dest   : /home/sceuser/RealEyes/gdrive/data_set_split

   train/FAKE: 3352 files  (0 newly copied)
   val  /FAKE:  419 files  (0 newly copied)
   test /FAKE:  419 files  (0 newly copied)
   train/REAL: 3345 files  (0 newly copied)
   val  /REAL:  418 files  (0 newly copied)
   test /REAL:  419 files  (0 newly copied)

✅ Split complete — 0 new files copied to /home/sceuser/RealEyes/gdrive/data_set_split
  train: 7,100 images
  val  : 1,061 images
  test : 1,061 images

☁️  Pushing data_set_split → Google Drive...


  ✅ data_set_split is now visible in Google Drive


In [4]:
# Cell 7
# =================== HELPER: LOAD DATASET IMAGES ===================

def load_dataset_images(dataset_path, max_images=None):
    """Load image paths and labels from a dataset directory with STRICT label assignment."""
    image_paths = []
    labels = []

    # ONLY these extensions are valid
    valid_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp'}

    folders = os.listdir(dataset_path)

    folders.sort()

    skipped_count = 0

    for folder in folders:
        path = os.path.join(dataset_path, folder)
        if not os.path.isdir(path):
            continue

        folder_upper = folder.upper()

        if folder_upper == 'FAKE':
            current_label = 1  # תמיד 1 לזיוף
        elif folder_upper == 'REAL':
            current_label = 0  # תמיד 0 לאמיתי
        else:
            print(f"⚠️ Warning: Unknown folder name '{folder}'. Skipping.")
            continue
        # =========================================================

        # Walk recursively so we find images in nested video subfolders (e.g. CelebDF structure:
        # REAL/video_name/frame_0000.jpg) as well as flat layouts (REAL/image.jpg).
        collected = []
        for root, _, files in os.walk(path):
            for filename in files:
                ext = os.path.splitext(filename)[1].lower()
                if ext not in valid_extensions:
                    skipped_count += 1
                    continue
                file_path = os.path.join(root, filename)
                mime_type, _ = mimetypes.guess_type(file_path)
                if mime_type and mime_type not in {'image/jpeg', 'image/png', 'image/gif', 'image/bmp'}:
                    skipped_count += 1
                    continue
                collected.append(file_path)

        if max_images:
            collected = collected[:max_images]

        for file_path in collected:
            image_paths.append(file_path)
            labels.append(current_label)

    if skipped_count > 0:
        print(f"   ⚠️ Skipped {skipped_count} non-standard files in {dataset_path}")

    print(f"   ✅ Loaded {len(image_paths)} valid images from {dataset_path}")
    return image_paths, np.array(labels)

In [5]:
# Cell 8
# =================== FULL DATA LOADING ===================

def load_all_datasets(datasets_dict, purpose_name):
    print(f"\n📦 Processing {purpose_name} Sets (FULL LOADING)...")
    print("=" * 60)

    all_paths = []
    all_labels = []

    for ds_name, ds_path in datasets_dict.items():
        print(f"   📥 Loading {ds_name} (ALL IMAGES)...")
        paths, labels = load_dataset_images(ds_path, max_images=None)
        all_paths.extend(paths)
        all_labels.extend(labels)

    return np.array(all_paths), np.array(all_labels)

print("🏗️ Building FULL Training Set...")
train_image_paths, train_labels = load_all_datasets(train_datasets, "TRAIN")

print("\n🏗️ Building FULL Validation Set...")
validation_image_paths, val_labels = load_all_datasets(validation_datasets, "VALIDATION")

print("\n🏗️ Building FULL Test Set...")
test_image_paths, test_labels = load_all_datasets(test_datasets, "TEST")

🏗️ Building FULL Training Set...

📦 Processing TRAIN Sets (FULL LOADING)...
   📥 Loading OpenForensics (ALL IMAGES)...
   ✅ Loaded 140002 valid images from /home/sceuser/RealEyes/RealEyes/datasets/OpenForensicsV1/Dataset/Train
   📥 Loading CustomWar (ALL IMAGES)...
   ✅ Loaded 7100 valid images from /home/sceuser/RealEyes/gdrive/data_set_split/train
   📥 Loading CiFake (ALL IMAGES)...
   ✅ Loaded 100000 valid images from /home/sceuser/RealEyes/RealEyes/datasets/cifake/train
   📥 Loading CelebDF (ALL IMAGES)...
   ✅ Loaded 71297 valid images from /home/sceuser/RealEyes/RealEyes/datasets/celebdf_v2/train

🏗️ Building FULL Validation Set...

📦 Processing VALIDATION Sets (FULL LOADING)...
   📥 Loading OpenForensics (ALL IMAGES)...
   ✅ Loaded 39428 valid images from /home/sceuser/RealEyes/RealEyes/datasets/OpenForensicsV1/Dataset/Validation
   📥 Loading CustomWar (ALL IMAGES)...
   ✅ Loaded 1061 valid images from /home/sceuser/RealEyes/gdrive/data_set_split/val
   📥 Loading CiFake (ALL IMA

In [27]:
# Cell 9
# =================== DATASET STATISTICS ===================

print("\n" + "=" * 60)
print("📊 FINAL FULL DATASET SUMMARY")
print("=" * 60)
print(f"Train: {len(train_image_paths):,}")
print(f"Val:   {len(validation_image_paths):,}")
print(f"Test:  {len(test_image_paths):,}")
print("=" * 60)

def print_label_stats(name, labels):
    labels = np.array(labels)
    real_count = np.sum(labels == 0)
    fake_count = np.sum(labels == 1)
    print(f"{name}: REAL={real_count:,}, FAKE={fake_count:,}, TOTAL={len(labels):,}")

print_label_stats("TRAIN", train_labels)
print_label_stats("VAL", val_labels)
print_label_stats("TEST", test_labels)


📊 FINAL FULL DATASET SUMMARY
Train: 318,399
Val:   68,285
Test:  18,841
TRAIN: REAL=132,429, FAKE=185,970, TOTAL=318,399
VAL: REAL=31,295, FAKE=36,990, TOTAL=68,285
TEST: REAL=8,337, FAKE=10,504, TOTAL=18,841


In [28]:
# Cell 10
##6# =================== SHARED PREPROCESSING ===================

IMG_SIZE_RGB = (224, 224)
IMG_SIZE_SRM = (256, 256)

BATCH_SIZE_RGB = 8
BATCH_SIZE_SRM = 32
BATCH_SIZE_HYBRID = 4

AUTOTUNE = tf.data.AUTOTUNE

def decode_rgb_image(path, label, img_size=(224, 224)):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, img_size)
    img = tf.cast(img, tf.float32)
    return img, tf.cast(label, tf.float32)

def decode_rgb_image_normalized(path, label, img_size=(256, 256)):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, img_size)
    img = tf.cast(img, tf.float32) / 255.0
    return img, tf.cast(label, tf.float32)

def create_rgb_dataset(image_paths, labels, batch_size=BATCH_SIZE_RGB, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    if shuffle:
        ds = ds.shuffle(min(len(image_paths), 10000), reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: decode_rgb_image(p, y, IMG_SIZE_RGB), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(1)
    return ds

print("✅ Shared preprocessing ready")

✅ Shared preprocessing ready


# SRM Model

In [6]:
# Cell 12
# =================== SRM FILTERS ===================

import numpy as np

# -------------------------------
# 5x5 SRM-style filters
# -------------------------------
filters_5x5 = [
    # Filter 1: Laplacian-High Boost
    [[0, 0, -1, 0, 0],
     [0, -1, 2, -1, 0],
     [-1, 2, 4, 2, -1],
     [0, -1, 2, -1, 0],
     [0, 0, -1, 0, 0]],

    # Filter 2: Edge & Noise Enhancer
    [[-1, 2, -2, 2, -1],
     [2, -6, 8, -6, 2],
     [-2, 8, -12, 8, -2],
     [2, -6, 8, -6, 2],
     [-1, 2, -2, 2, -1]],

    # Filter 3: Diagonal Residuals
    [[2, -1, 0, -1, 2],
     [-1, -2, 3, -2, -1],
     [0, 3, 0, 3, 0],
     [-1, -2, 3, -2, -1],
     [2, -1, 0, -1, 2]],

    # Filter 4: Vertical Edge
    [[0, 0, 0, 0, 0],
     [1, -2, 1, -2, 1],
     [0, 0, 0, 0, 0],
     [-1, 2, -1, 2, -1],
     [0, 0, 0, 0, 0]],

    # Filter 5: High-Frequency / Noise Pattern
    [[1, -4, 6, -4, 1],
     [-4, 16, -24, 16, -4],
     [6, -24, 36, -24, 6],
     [-4, 16, -24, 16, -4],
     [1, -4, 6, -4, 1]],
]

# -------------------------------
# 3x3 filters to be padded into 5x5
# -------------------------------
filters_3x3_raw = [
    [[0, -1, 0],
     [-1, 4, -1],
     [0, -1, 0]],  # Basic Laplacian

    [[-1, 2, -1],
     [2, -4, 2],
     [-1, 2, -1]],  # Residual

    [[-1, -1, -1],
     [-1, 8, -1],
     [-1, -1, -1]]  # Point detection
]

def pad_3x3_to_5x5(kernels_3x3):
    padded_kernels = []
    for k in kernels_3x3:
        padded = np.pad(k, ((1, 1), (1, 1)), mode='constant', constant_values=0)
        padded_kernels.append(padded)
    return padded_kernels

filters_3x3_padded = pad_3x3_to_5x5(filters_3x3_raw)

# -------------------------------
# Final filter bank: 8 filters total
# -------------------------------
all_filters = np.array(filters_5x5 + filters_3x3_padded, dtype=np.float32)

print(f"✅ SRM filters ready: {len(all_filters)} filters, each of size 5x5")
print("all_filters shape:", all_filters.shape)

✅ SRM filters ready: 8 filters, each of size 5x5
all_filters shape: (8, 5, 5)


In [7]:
# Cell 13
# =================== SRM FILTER FUNCTION ===================

import tensorflow as tf

# all_filters shape: (8, 5, 5)
# TF kernel shape should be: (5, 5, 1, 8)
srm_filters_tf = tf.constant(
    np.transpose(all_filters[:, :, :, np.newaxis], (1, 2, 3, 0)),
    dtype=tf.float32
)

def apply_srm_filters_tf(image):
    """
    Apply SRM filters (8 filters + tanh) to an RGB image.

    Args:
        image: Tensor of shape (H, W, 3) or (B, H, W, 3)

    Returns:
        SRM feature maps of shape:
        - (256, 256, 24) for single image
        - (B, 256, 256, 24) for batch
    """
    if len(image.shape) == 3:
        image = image[tf.newaxis, ...]
        squeeze = True
    else:
        squeeze = False

    image = tf.image.resize(image, [256, 256])
    image = tf.cast(image, tf.float32)

    channels = tf.split(image, num_or_size_splits=3, axis=-1)

    feature_maps = []
    for channel in channels:
        fm = tf.nn.conv2d(channel, srm_filters_tf, strides=1, padding='SAME')
        fm = (tf.math.tanh(fm) + 1.0) / 2.0
        feature_maps.append(fm)

    result = tf.concat(feature_maps, axis=-1)

    if squeeze:
        result = result[0]

    return result

print("✅ apply_srm_filters_tf() ready")
print("   Input: RGB image (H, W, 3) or (B, H, W, 3)")
print("   Output: SRM feature maps (256, 256, 24) or (B, 256, 256, 24)")
print("   Kernel tensor shape:", srm_filters_tf.shape)

✅ apply_srm_filters_tf() ready
   Input: RGB image (H, W, 3) or (B, H, W, 3)
   Output: SRM feature maps (256, 256, 24) or (B, 256, 256, 24)
   Kernel tensor shape: (5, 5, 1, 8)


I0000 00:00:1776766612.110110 3205271 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13775 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5


In [31]:
# Cell 14
##9# =================== SRM DATASET PIPELINE ===================

def srm_augment(srm_features, label):
    srm_features = tf.image.random_flip_left_right(srm_features)
    srm_features = tf.image.random_flip_up_down(srm_features)
    srm_features = tf.image.random_brightness(srm_features, max_delta=0.05)
    srm_features = tf.clip_by_value(srm_features, 0.0, 1.0)
    return srm_features, label

def create_srm_dataset(image_paths, labels, batch_size=BATCH_SIZE_SRM, shuffle=False, augment=False):
    def process_image(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, [256, 256])
        img = tf.cast(img, tf.float32) / 255.0

        srm_features = apply_srm_filters_tf(img)
        srm_features = tf.ensure_shape(srm_features, [256, 256, 24])

        return srm_features, tf.cast(label, tf.float32)

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    if shuffle:
        ds = ds.shuffle(min(len(image_paths), 10000), reshuffle_each_iteration=True)

    ds = ds.map(process_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(srm_augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(1)

    return ds

print("✅ SRM dataset pipeline ready")

✅ SRM dataset pipeline ready


In [32]:
# Cell 15
# =================== PREPARE FULL SRM DATASETS ===================

print("📦 Creating FULL SRM datasets...")
print("=" * 60)

train_srm_dataset = create_srm_dataset(
    train_image_paths,
    train_labels,
    batch_size=BATCH_SIZE_SRM,
    shuffle=True,
    augment=True
)

val_srm_dataset = create_srm_dataset(
    validation_image_paths,
    val_labels,
    batch_size=BATCH_SIZE_SRM,
    shuffle=False
)

test_srm_dataset = create_srm_dataset(
    test_image_paths,
    test_labels,
    batch_size=BATCH_SIZE_SRM,
    shuffle=False
)

print(f"✅ Train SRM images: {len(train_image_paths):,}")
print(f"✅ Val SRM images:   {len(validation_image_paths):,}")
print(f"✅ Test SRM images:  {len(test_image_paths):,}")

📦 Creating FULL SRM datasets...


✅ Train SRM images: 318,399
✅ Val SRM images:   68,285
✅ Test SRM images:  18,841


In [33]:
# Cell 16
# =================== SRM SANITY CHECK ===================

for x_batch, y_batch in train_srm_dataset.take(1):
    print("SRM batch shape:", x_batch.shape)
    print("Labels shape:", y_batch.shape)
    print("Min:", tf.reduce_min(x_batch).numpy(), "Max:", tf.reduce_max(x_batch).numpy())

SRM batch shape: (32, 256, 256, 24)
Labels shape: (32,)
Min: 0.0 Max: 1.0


2026-04-11 13:22:43.034392: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 335543808 bytes after encountering the first element of size 335543808 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
2026-04-11 13:22:43.080701: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [34]:
# Cell 17
# =================== CLEAR GPU MEMORY ===================

import tensorflow as tf
import gc
import os

print("TensorFlow version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

gc.collect()
tf.keras.backend.clear_session()

print("✅ Cleared previous session")


print("📊 GPU MEMORY STATUS:")
os.system("nvidia-smi")

TensorFlow version: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ Cleared previous session
📊 GPU MEMORY STATUS:
Sat Apr 11 13:22:43 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   72C    P0              32W /  70W |  13932MiB / 15360MiB |      0%      Default |
|         

0

In [35]:
# Cell 18
# =================== IMPROVED AUTOENCODER / ENCODER ===================
# IMPROVEMENTS over original:
#   1. SE (Squeeze-and-Excitation) attention blocks -> model learns which noise channels matter
#   2. Residual connections -> better gradient flow, deeper effective network
#   3. L2 regularization on all conv layers -> reduces overfitting in downstream classifier
#   4. Fixed decoder activation: sigmoid (was tanh -- mismatch with [0,1] SRM inputs)
#   5. Proper BN momentum/epsilon for more stable training
#
# CURRENT APPROACH (v1): SRM input 256x256x24 -> Encoder latent 16x16x256
#   Training status: [FILL AFTER TRAINING - epochs completed, final val_loss]
#   Reconstruction val MSE: [FILL AFTER TRAINING]
#   Note: 256x256 is compatible with EfficientNetB4 (v2 upgrade) because
#   streams merge after GlobalAveragePooling so spatial sizes need not match
#
# CURRENT APPROACH (v1): SRM input 256x256x24 -> Encoder latent 16x16x256
#   - Training status: [FILL AFTER TRAINING -- epochs completed, final val_loss]
#   - Reconstruction quality (val MSE): [FILL AFTER TRAINING]
#   - Note: 256x256 is compatible with EfficientNetB4 upgrade because streams
#     merge after GlobalAveragePooling -- spatial size does not need to match

from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import tensorflow as tf
import os

REG = tf.keras.regularizers.l2(1e-4)

def conv_bn_lrelu(x, filters, kernel=3, name=""):
    x = layers.Conv2D(filters, kernel, padding="same", use_bias=False,
                      kernel_regularizer=REG, name=f"{name}_conv")(x)
    x = layers.BatchNormalization(momentum=0.9, epsilon=1e-5, name=f"{name}_bn")(x)
    x = layers.LeakyReLU(negative_slope=0.1, name=f"{name}_act")(x)
    return x

def se_block(x, ratio=16, name=""):
    """Channel attention: squeeze global context, excite important channels."""
    ch = x.shape[-1]
    r = max(1, ch // ratio)
    se = layers.GlobalAveragePooling2D(keepdims=True, name=f"{name}_gap")(x)
    se = layers.Dense(r, activation="relu",  use_bias=False, name=f"{name}_fc1")(se)
    se = layers.Dense(ch, activation="sigmoid", use_bias=False, name=f"{name}_fc2")(se)
    return layers.Multiply(name=f"{name}_scale")([x, se])

def residual_se_block(x, filters, name=""):
    """Conv → BN → LReLU → Conv → BN → SE → Add shortcut."""
    shortcut = x
    x = conv_bn_lrelu(x, filters, name=f"{name}_a")
    x = conv_bn_lrelu(x, filters, name=f"{name}_b")
    x = se_block(x, ratio=16, name=f"{name}_se")
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding="same", use_bias=False,
                                  kernel_regularizer=REG, name=f"{name}_proj")(shortcut)
    return layers.Add(name=f"{name}_add")([x, shortcut])

print("🔨 Building Improved SRM Autoencoder (SE + Residual)...")

input_img = layers.Input(shape=(256, 256, 24), name="srm_input")

# ===========================
# Encoder  (256→128→64→32→16)
# ===========================
x = conv_bn_lrelu(input_img, 32, name="enc_s1")
x = residual_se_block(x, 32, name="enc_res1")
x = layers.MaxPooling2D((2, 2), name="enc_pool1")(x)          # 128×128×32

x = conv_bn_lrelu(x, 64, name="enc_s2")
x = residual_se_block(x, 64, name="enc_res2")
x = layers.MaxPooling2D((2, 2), name="enc_pool2")(x)          # 64×64×64

x = conv_bn_lrelu(x, 128, name="enc_s3")
x = residual_se_block(x, 128, name="enc_res3")
x = layers.MaxPooling2D((2, 2), name="enc_pool3")(x)          # 32×32×128

x = conv_bn_lrelu(x, 256, name="enc_s4")
x = residual_se_block(x, 256, name="enc_res4")
encoded = layers.MaxPooling2D((2, 2), name="encoded_latent")(x)  # 16×16×256  ← latent space

# ===========================
# Decoder  (16→32→64→128→256)
# ===========================
x = conv_bn_lrelu(encoded, 256, name="dec_s1")
x = layers.UpSampling2D((2, 2), name="dec_up1")(x)            # 32×32

x = conv_bn_lrelu(x, 128, name="dec_s2")
x = layers.UpSampling2D((2, 2), name="dec_up2")(x)            # 64×64

x = conv_bn_lrelu(x, 64, name="dec_s3")
x = layers.UpSampling2D((2, 2), name="dec_up3")(x)            # 128×128

x = conv_bn_lrelu(x, 32, name="dec_s4")
x = layers.UpSampling2D((2, 2), name="dec_up4")(x)            # 256×256

decoded = layers.Conv2D(
    24, (3, 3), activation="sigmoid",   # sigmoid → output in [0,1] matches SRM feature range
    padding="same", name="decoder_output"
)(x)

# --------------------------------------------------
# Models
# --------------------------------------------------
autoencoder = Model(input_img, decoded, name="srm_autoencoder_24ch")
encoder     = Model(input_img, encoded, name="srm_encoder_24ch")

autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss="mse"
)

print("✅ Improved Autoencoder built successfully")
print("Autoencoder output shape:", autoencoder.output_shape)
print("Encoder output shape:    ", encoder.output_shape)

🔨 Building Improved SRM Autoencoder (SE + Residual)...


✅ Improved Autoencoder built successfully
Autoencoder output shape: (None, 256, 256, 24)
Encoder output shape:     (None, 16, 16, 256)


In [36]:
# Cell 19
##14# =================== AUTOENCODER SANITY CHECK ===================
# Wrapped in try/except — a JIT/GPU error here must NOT block the rest of the training.

try:
    for x_batch, _ in train_srm_dataset.take(1):
        recon  = autoencoder(x_batch, training=False)
        latent = encoder(x_batch, training=False)
        print("Input batch shape:   ", x_batch.shape)
        print("Reconstruction shape:", recon.shape)
        print("Latent shape:        ", latent.shape)
        break
    print("✅ Sanity check passed")
except Exception as e:
    print(f"⚠️  Sanity check skipped (non-fatal): {type(e).__name__}: {str(e)[:200]}")
    print("   Training will continue — the error does not affect model correctness.")

2026-04-11 13:22:44.588782: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 268435712 bytes after encountering the first element of size 268435712 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


Input batch shape:    (32, 256, 256, 24)
Reconstruction shape: (32, 256, 256, 24)
Latent shape:         (32, 16, 16, 256)
✅ Sanity check passed


In [37]:
# Cell 20
# =================== PREPARE AUTOENCODER DATASETS ===================

ae_train_ds = train_srm_dataset.map(lambda x, y: (x, x), num_parallel_calls=tf.data.AUTOTUNE).prefetch(1)
ae_val_ds   = val_srm_dataset.map(lambda x, y: (x, x), num_parallel_calls=tf.data.AUTOTUNE).prefetch(1)

print("✅ Autoencoder datasets ready")

✅ Autoencoder datasets ready


In [ ]:
# Cell 21
# =================== AUTOENCODER TRAINING ===================

ae_checkpoint_path = os.path.join(AE_DIR, "best_autoencoder_24ch.keras")

_existing_ae = find_latest_model("final_autoencoder_24ch.keras") if SKIP_IF_SAVED else None
if _existing_ae:
    print(f"⏭️  SKIP_IF_SAVED=True — loading autoencoder from:\n   {_existing_ae}")
    autoencoder = tf.keras.models.load_model(_existing_ae, compile=False)
    autoencoder.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    encoder = tf.keras.Model(
        inputs=autoencoder.input,
        outputs=autoencoder.get_layer("encoded_latent").output,
        name="srm_encoder_24ch"
    )
    print("✅ Autoencoder + encoder loaded from saved checkpoint")
else:
    ae_callbacks = [
        EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
        ModelCheckpoint(filepath=ae_checkpoint_path, monitor="val_loss",
                        save_best_only=True, verbose=1),
    ]
    print("🚀 Starting Autoencoder training...")
    history_ae = autoencoder.fit(
        ae_train_ds, validation_data=ae_val_ds,
        epochs=30, callbacks=ae_callbacks, verbose=1
    )
    print("✅ Autoencoder training complete")

🚀 Starting Autoencoder training...
Epoch 1/30


4178/9950 ━━━━━━━━━━━━━━━━━━━━ 48:08 500ms/step - loss: 0.0185

2026-04-11 13:57:56.995992: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 107956818 exceeds 10% of free system memory.


4187/9950 ━━━━━━━━━━━━━━━━━━━━ 48:03 500ms/step - loss: 0.0184

2026-04-11 13:58:01.676087: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 66355200 exceeds 10% of free system memory.


...

9949/9950 ━━━━━━━━━━━━━━━━━━━━ 0s 500ms/step - loss: 0.0109

2026-04-11 14:46:13.518358: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[31,24,256,256]{3,2,1,0}, u8[0]{0}) custom-call(f32[31,32,256,256]{3,2,1,0}, f32[24,32,3,3]{3,2,1,0}, f32[24]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]}
2026-04-11 14:46:18.125724: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-11 14:46:18.405032: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measur

9950/9950 ━━━━━━━━━━━━━━━━━━━━ 0s 505ms/step - loss: 0.0109

2026-04-11 15:00:31.727819: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[29,24,256,256]{3,2,1,0}, u8[0]{0}) custom-call(f32[29,32,256,256]{3,2,1,0}, f32[24,32,3,3]{3,2,1,0}, f32[24]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]}



Epoch 1: val_loss improved from None to 0.00476, saving model to /home/sceuser/RealEyes/gdrive/deepfake_image_project/models/RealEyes_v2/autoencoder/best_autoencoder_24ch.keras

Epoch 1: finished saving model to /home/sceuser/RealEyes/gdrive/deepfake_image_project/models/RealEyes_v2/autoencoder/best_autoencoder_24ch.keras
9950/9950 ━━━━━━━━━━━━━━━━━━━━ 5871s 588ms/step - loss: 0.0047 - val_loss: 0.0048 - learning_rate: 5.0000e-04
Epoch 2/30
9950/9950 ━━━━━━━━━━━━━━━━━━━━ 0s 500ms/step - loss: 0.0034
Epoch 2: val_loss did not improve from 0.00476
9950/9950 ━━━━━━━━━━━━━━━━━━━━ 5786s 581ms/step - loss: 0.0031 - val_loss: 0.0053 - learning_rate: 5.0000e-04
Epoch 3/30
9950/9950 ━━━━━━━━━━━━━━━━━━━━ 0s 497ms/step - loss: 0.0032
Epoch 3: val_loss improved from 0.00476 to 0.00399, saving model to /home/sceuser/RealEyes/gdrive/deepfake_image_project/models/RealEyes_v2/autoencoder/best_autoencoder_24ch.keras

Epoch 3: finished saving model to /home/sceuser/RealEyes/gdrive/deepfake_image_projec

In [ ]:
# Cell 22
# =================== SAVE / LOAD ENCODER ===================
# Only save when we actually trained (not when we skipped by loading)

if not _existing_ae:
    save_model_versioned(encoder,     AE_DIR, "final_encoder_24ch")
    save_model_versioned(autoencoder, AE_DIR, "final_autoencoder_24ch")

encoder_path = find_latest_model("final_encoder_24ch.keras")
print("Encoder output shape:", encoder.output_shape)
print("✅ Encoder path:", encoder_path)

In [ ]:
# Cell 23
# =================== CLEAR MEMORY BEFORE CLASSIFIER ===================

import gc
gc.collect()
tf.keras.backend.clear_session()

print("✅ Memory cleared")

In [ ]:
# Cell 24
# =================== CNN-SRM CLASSIFIER ===================

from tensorflow.keras import layers, Model
import tensorflow as tf
import os

print("🔨 Building CNN-SRM classifier on encoder features...")

L2 = tf.keras.regularizers.l2(1e-4)

# --------------------------------------------------
# Phase 1: freeze encoder
# --------------------------------------------------
encoder.trainable = False

# IMPROVED classifier head:
#   - Removed extra conv layers on the latent (they added parameters without benefit)
#   - GlobalAveragePooling directly on encoder output, then Dense layers
#   - Added L2 regularization and higher dropout to fight severe overfitting
classifier_input = layers.Input(shape=encoder.output_shape[1:], name="latent_input")

x = layers.GlobalAveragePooling2D(name="cls_gap")(classifier_input)

x = layers.Dense(256, kernel_regularizer=L2, name="cls_dense1")(x)
x = layers.BatchNormalization(name="cls_bn1")(x)
x = layers.Activation("relu", name="cls_act1")(x)
x = layers.Dropout(0.5, name="cls_drop1")(x)

x = layers.Dense(128, kernel_regularizer=L2, name="cls_dense2")(x)
x = layers.BatchNormalization(name="cls_bn2")(x)
x = layers.Activation("relu", name="cls_act2")(x)
x = layers.Dropout(0.4, name="cls_drop2")(x)

x = layers.Dense(64, kernel_regularizer=L2, name="cls_dense3")(x)
x = layers.Activation("relu", name="cls_act3")(x)
x = layers.Dropout(0.3, name="cls_drop3")(x)

output = layers.Dense(1, activation="sigmoid", name="prob_fake")(x)

classifier_head = Model(classifier_input, output, name="latent_classifier_head")

# full model = SRM input -> encoder -> classifier head
full_input = layers.Input(shape=(256, 256, 24), name="srm_input")
latent = encoder(full_input)
pred = classifier_head(latent)

cnn_srm = Model(full_input, pred, name="SRM_Encoder_Classifier")

cnn_srm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),  # label smoothing reduces overconfidence
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

print("✅ CNN-SRM classifier built successfully")
print("Full model input shape:", cnn_srm.input_shape)
print("Full model output shape:", cnn_srm.output_shape)

In [ ]:
# Cell 25
# =================== CNN-SRM SANITY CHECK ===================

for x_batch, y_batch in train_srm_dataset.take(1):
    x_one = x_batch[:1]
    y_one = y_batch[:1]

    pred = cnn_srm(x_one, training=False)

    print("Input shape:", x_one.shape)
    print("Label shape:", y_one.shape)
    print("Prediction shape:", pred.shape)
    print("Prediction value:", pred.numpy())
    break

In [8]:
# Cell 26
# =================== CLASS WEIGHTS (must be before any training) ===================
# FIX: was originally defined AFTER the training cells — moved here to avoid NameError
#
# Manual weights instead of compute_class_weight("balanced"):
# "balanced" was giving FAKE (label=1) a weight of ~0.86 and REAL (label=0) ~1.20
# because FAKE is the majority class — the opposite of what we want.
# Setting FAKE weight higher forces all three models (CNN-SRM, EfficientNetB0,
# Two-Stream) to penalise missed FAKEs more strongly and reduces REAL-prediction bias.

srm_class_weights = {0: 1.0, 1: 2.5}

print("✅ Class weights ready:", srm_class_weights)

✅ Class weights ready: {0: 1.0, 1: 2.5}


In [ ]:
# Cell 27
# =================== CNN-SRM TRAINING - PHASE 1 ===================

phase1_checkpoint = os.path.join(CNN_SRM_DIR, "best_cnn_srm_phase1.keras")

_existing_cnn = find_latest_model("final_cnn_srm.keras") if SKIP_IF_SAVED else None
if _existing_cnn:
    print(f"⏭️  SKIP_IF_SAVED=True — loading CNN-SRM from:\n   {_existing_cnn}")
    cnn_srm = tf.keras.models.load_model(_existing_cnn, compile=False)
    cnn_srm.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    print("✅ CNN-SRM loaded — skipping Phase 1 & 2 training")
else:
    callbacks_phase1 = [
        tf.keras.callbacks.EarlyStopping(monitor="val_auc", patience=4,
                                          restore_best_weights=True, mode="max", verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", factor=0.5,
                                              patience=2, mode="max", verbose=1),
        tf.keras.callbacks.ModelCheckpoint(filepath=phase1_checkpoint, monitor="val_auc",
                                            save_best_only=True, mode="max", verbose=1),
    ]
    print("🚀 Starting CNN-SRM Phase 1 training (encoder frozen)...")
    history_phase1 = cnn_srm.fit(
        train_srm_dataset, validation_data=val_srm_dataset,
        epochs=15, class_weight=srm_class_weights,
        callbacks=callbacks_phase1, verbose=1
    )
    print("✅ Phase 1 complete")

In [ ]:
# Cell 28
# =================== CNN-SRM TRAINING - PHASE 2 ===================

if _existing_cnn:
    print("⏭️  Phase 2 skipped — model was loaded from saved checkpoint")
else:
    print("🔓 Unfreezing top encoder layers for fine-tuning...")
    encoder.trainable = True
    fine_tune_from = int(len(encoder.layers) * 0.70)
    for layer in encoder.layers[:fine_tune_from]:
        layer.trainable = False
    print(f"Encoder layers: {len(encoder.layers)} | Frozen: {fine_tune_from} | Trainable: {len(encoder.layers)-fine_tune_from}")

    phase2_checkpoint = os.path.join(CNN_SRM_DIR, "best_cnn_srm_finetuned.keras")
    cnn_srm.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5, clipnorm=1.0),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    callbacks_phase2 = [
        tf.keras.callbacks.EarlyStopping(monitor="val_auc", patience=5,
                                          restore_best_weights=True, mode="max", verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", factor=0.5,
                                              patience=2, mode="max", verbose=1),
        tf.keras.callbacks.ModelCheckpoint(filepath=phase2_checkpoint, monitor="val_auc",
                                            save_best_only=True, mode="max", verbose=1),
    ]
    print("🚀 Starting CNN-SRM Phase 2 fine-tuning...")
    history_phase2 = cnn_srm.fit(
        train_srm_dataset, validation_data=val_srm_dataset,
        epochs=20, class_weight=srm_class_weights,
        callbacks=callbacks_phase2, verbose=1
    )
    print("✅ Phase 2 complete")

In [ ]:
# Cell 29
# =================== CNN-SRM EVALUATION ===================

test_results = cnn_srm.evaluate(test_srm_dataset, verbose=1)

print("\n📊 CNN-SRM Test Results:")
for name, value in zip(cnn_srm.metrics_names, test_results):
    print(f"{name}: {value:.4f}")

In [ ]:
# Cell 30
# =================== CNN-SRM DETAILED EVALUATION ===================
# FIX: was a duplicate of the previous cell — repurposed as detailed per-class metrics

import numpy as np
from sklearn.metrics import classification_report, roc_auc_score

y_true_srm, y_prob_srm = [], []
for x_batch, y_batch in test_srm_dataset:
    preds = cnn_srm.predict(x_batch, verbose=0).flatten()
    y_true_srm.extend(y_batch.numpy())
    y_prob_srm.extend(preds)

y_true_srm = np.array(y_true_srm).astype(int)
y_prob_srm = np.array(y_prob_srm)
y_pred_srm = (y_prob_srm > 0.5).astype(int)

print("📊 CNN-SRM Classification Report:")
print(classification_report(y_true_srm, y_pred_srm, target_names=["REAL", "FAKE"], digits=4))
print(f"ROC-AUC: {roc_auc_score(y_true_srm, y_prob_srm):.4f}")

In [ ]:
# Cell 31
# =================== SAVE CNN-SRM MODEL ===================

if not _existing_cnn:
    save_model_versioned(cnn_srm, CNN_SRM_DIR, "final_cnn_srm")

cnn_srm_path = find_latest_model("final_cnn_srm.keras")
print("✅ CNN-SRM model at:", cnn_srm_path)

In [ ]:
# Cell 32
# ============================================================================
# CNN-SRM Visualizations (Confusion Matrix + ROC + Metrics Table)
# מותאם למחברת שלך: cnn_srm + test_srm_dataset
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report, roc_auc_score

# ============================================================================
# 1. Collect predictions from cnn_srm on test_srm_dataset
# ============================================================================

y_true = []
y_prob = []

for x_batch, y_batch in test_srm_dataset:
    preds = cnn_srm.predict(x_batch, verbose=0).flatten()
    y_true.extend(y_batch.numpy())
    y_prob.extend(preds)

y_true = np.array(y_true).astype(int)
y_prob = np.array(y_prob)
y_pred_binary = (y_prob > 0.5).astype(int)

print("✅ Predictions collected successfully")
print("y_true shape:", y_true.shape)
print("y_prob shape:", y_prob.shape)
print("First 10 probabilities:", y_prob[:10])

# ============================================================================
# 2. Compute metrics
# ============================================================================

cm = confusion_matrix(y_true, y_pred_binary)
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

clf_report = classification_report(
    y_true,
    y_pred_binary,
    target_names=['REAL', 'FAKE'],
    output_dict=True
)

report_df = pd.DataFrame(clf_report).T
report_df = report_df.loc[
    ['FAKE', 'REAL', 'macro avg', 'weighted avg'],
    ['precision', 'recall', 'f1-score', 'support']
]

# ============================================================================
# 3. Plot: Confusion Matrix + ROC + Classification Table
# ============================================================================

plt.style.use('default')
sns.set_palette("husl")

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# ========== 1. Confusion Matrix ==========
ax1 = axes[0]

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    ax=ax1,
    xticklabels=['REAL', 'FAKE'],
    yticklabels=['REAL', 'FAKE'],
    cbar_kws={'label': 'Count'}
)

ax1.set_xlabel('Predicted', fontsize=12)
ax1.set_ylabel('True', fontsize=12)
ax1.set_title('CNN-SRM Confusion Matrix', fontsize=14, fontweight='bold')

# ========== 2. ROC Curve ==========
ax2 = axes[1]

ax2.plot(
    fpr, tpr,
    color='darkorange',
    linewidth=2,
    label=f'ROC Curve (AUC = {roc_auc:.4f})'
)
ax2.plot(
    [0, 1], [0, 1],
    color='navy',
    linewidth=2,
    linestyle='--',
    label='Random'
)

ax2.set_xlabel('False Positive Rate', fontsize=12)
ax2.set_ylabel('True Positive Rate', fontsize=12)
ax2.set_title('CNN-SRM ROC Curve', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

# ========== 3. Classification Metrics Table ==========
ax3 = axes[2]

sns.heatmap(
    report_df,
    annot=True,
    cmap='Blues',
    fmt='d',
    ax=ax3,
    cbar=False
)

ax3.set_title('CNN-SRM Classification Metrics', fontsize=14, fontweight='bold')
ax3.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

# ============================================================================
# 4. Text report
# ============================================================================

print("\n" + "=" * 70)
print("📋 CNN-SRM CLASSIFICATION REPORT")
print("=" * 70)
print(classification_report(y_true, y_pred_binary, target_names=['REAL', 'FAKE'], digits=4))

print(f"\n✅ CNN-SRM ROC AUC: {roc_auc:.4f}")
print("✅ Visualizations complete!")

# EfficientNetB0 Model

In [9]:
# Cell 34
import tensorflow as tf
import numpy as np
import os
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.metrics import AUC, Precision, Recall

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def decode_and_resize(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, tf.cast(label, tf.float32)

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.15)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.85, upper=1.15)
    image = tf.image.random_hue(image, max_delta=0.05)

    # FIX: tf.keras.layers.RandomRotation() must NOT be instantiated inside map().
    # Each call to map() would create a new Keras layer object — expensive and causes graph
    # tracing issues. Use tf.raw_ops or simply skip rotation (flips already cover most needs).
    # Lightweight crop-and-resize trick gives implicit translation/scale robustness instead:
    image = tf.image.random_crop(image, size=[IMG_SIZE[0], IMG_SIZE[1], 3])

    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label

def preprocess_for_efficientnet(image, label):
    image = preprocess_input(image)
    return image, label

def build_rgb_dataset(image_paths, labels, batch_size=16, training=False):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    if training:
        ds = ds.shuffle(min(len(image_paths), 10000), reshuffle_each_iteration=True)

    ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)

    ds = ds.map(preprocess_for_efficientnet, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

train_rgb_ds = build_rgb_dataset(train_image_paths, train_labels, batch_size=BATCH_SIZE, training=True)
val_rgb_ds   = build_rgb_dataset(validation_image_paths, val_labels, batch_size=BATCH_SIZE, training=False)
test_rgb_ds  = build_rgb_dataset(test_image_paths, test_labels, batch_size=BATCH_SIZE, training=False)

print("✅ RGB datasets ready")
print("Train samples:", len(train_image_paths))
print("Val samples:", len(validation_image_paths))
print("Test samples:", len(test_image_paths))

✅ RGB datasets ready
Train samples: 318399
Val samples: 68285
Test samples: 18841


In [10]:
# Cell 35
def build_efficientnet_b0(input_shape=(224, 224, 3), train_base=False):
    base_model = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape
    )
    base_model.trainable = train_base

    inputs = Input(shape=input_shape, name="rgb_input")
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.30)(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.25)(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.15)(x)
    outputs = Dense(1, activation="sigmoid", name="prob_fake")(x)

    model = Model(inputs, outputs, name="EfficientNetB0_Deepfake")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            "accuracy",
            AUC(name="auc"),
            Precision(name="precision"),
            Recall(name="recall")
        ]
    )
    return model, base_model

eff_model, eff_base = build_efficientnet_b0(train_base=False)
eff_model.summary()

Model: "EfficientNetB0_Deepfake"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rgb_input (InputLayer)          │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ prob_fake (Dense)               │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,394,020 (16.76 MB)

 Trainable params: 344,449 (1.31 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [11]:
# Cell 36
# NOTE: class weights were MOVED to cell before CNN-SRM Phase 1 training (to fix NameError).
# srm_class_weights is already defined — just confirming value here.
print("✅ srm_class_weights (already computed above):", srm_class_weights)

✅ srm_class_weights (already computed above): {0: 1.0, 1: 2.5}


In [ ]:
# Cell 37
SAVE_DIR    = EFF_DIR
stage1_path = os.path.join(EFF_DIR, "efficientnetb0_stage1_best.keras")

_existing_eff = find_latest_model("efficientnetb0_finetuned_best.keras") if SKIP_IF_SAVED else None
if _existing_eff:
    print(f"⏭️  SKIP_IF_SAVED=True — EfficientNet already trained. Stage 1 skipped.")
else:
    callbacks_stage1 = [
        ModelCheckpoint(stage1_path, monitor="val_auc", mode="max", save_best_only=True, verbose=1),
        EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ]
    history_stage1 = eff_model.fit(
        train_rgb_ds, validation_data=val_rgb_ds,
        epochs=15, class_weight=srm_class_weights,
        callbacks=callbacks_stage1
    )
    push_models_to_gdrive()

In [12]:
# Cell 38
SAVE_DIR    = EFF_DIR
stage1_path = os.path.join(EFF_DIR, "efficientnetb0_stage1_best.keras")
stage2_path = os.path.join(EFF_DIR, "efficientnetb0_finetuned_best.keras")

if "_existing_eff" not in dir():
    _existing_eff = find_latest_model("efficientnetb0_finetuned_best.keras") if SKIP_IF_SAVED else None

if _existing_eff:
    print(f"⏭️  Loading EfficientNet from:\n   {_existing_eff}")
    eff_model   = tf.keras.models.load_model(_existing_eff, compile=False)
    stage2_path = _existing_eff
    print("✅ EfficientNet loaded — skipping Stage 2 training")
else:
    eff_model = tf.keras.models.load_model(stage1_path)
    eff_base  = eff_model.get_layer("efficientnetb0")
    eff_base.trainable = True
    fine_tune_at = int(len(eff_base.layers) * 0.7)
    for layer in eff_base.layers[:fine_tune_at]:
        layer.trainable = False
    eff_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5, clipnorm=1.0),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05),
        metrics=["accuracy", AUC(name="auc"), Precision(name="precision"), Recall(name="recall")]
    )
    callbacks_stage2 = [
        ModelCheckpoint(stage2_path, monitor="val_auc", mode="max", save_best_only=True, verbose=1),
        EarlyStopping(monitor="val_auc", mode="max", patience=6, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ]
    print("🚀 EfficientNet Stage 2 fine-tuning...")
    history_stage2 = eff_model.fit(
        train_rgb_ds, validation_data=val_rgb_ds,
        epochs=15, class_weight=srm_class_weights,
        callbacks=callbacks_stage2
    )
    push_models_to_gdrive()
    print("✅ Stage 2 complete")

NameError: name '_existing_eff' is not defined

In [ ]:
# Cell 39
best_eff_model = tf.keras.models.load_model(stage2_path)

test_results = best_eff_model.evaluate(test_rgb_ds, verbose=1)
print("\n✅ EfficientNetB0 Test Results:")
for name, value in zip(best_eff_model.metrics_names, test_results):
    print(f"  {name}: {value:.4f}")

In [ ]:
# Cell 40
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import numpy as np

y_true = []
y_prob = []

for x_batch, y_batch in test_rgb_ds:
    probs = best_eff_model.predict(x_batch, verbose=0).flatten()
    y_true.extend(y_batch.numpy())
    y_prob.extend(probs)

y_true = np.array(y_true).astype(int)
y_prob = np.array(y_prob)
y_pred = (y_prob > 0.5).astype(int)  # FIX: was 0.2 — a lowered threshold is a symptom of miscalibration;
                                      # correct the model (label smoothing + LR fix) rather than hacking the threshold

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))

print("\nROC AUC:")
print(roc_auc_score(y_true, y_prob))

In [ ]:
# Cell 41


# Two-Stream Model
Combines the **CNN-SRM stream** (noise/frequency patterns via the pre-trained encoder) with the **EfficientNetB0 stream** (semantic RGB appearance) into a single joint classifier.

Architecture:
- **Stream 1 (noise)**: SRM input → frozen encoder → GAP → 256-dim feature vector  
- **Stream 2 (RGB)**: RGB input → frozen EfficientNetB0 backbone → GAP → 256-dim feature vector  
- **Fusion**: Concatenate both vectors → Dense head → binary output

In [ ]:
# Cell 43
# =================== TWO-STREAM DATASET PIPELINE ===================
# Each sample produces TWO inputs from the same image:
#   - SRM features  (256, 256, 24)  for the noise/frequency stream
#   - RGB features  (224, 224,  3)  for the EfficientNetB0 stream

import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess

TWOSTREAM_BATCH = 16   # smaller batch — two large tensors per sample

def process_dual_stream(path, label):
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img = tf.cast(img, tf.float32)

    # --- Stream 1: SRM ---
    img_srm = tf.image.resize(img, [256, 256]) / 255.0
    srm_feat = apply_srm_filters_tf(img_srm)
    srm_feat = tf.ensure_shape(srm_feat, [256, 256, 24])

    # --- Stream 2: RGB (EfficientNet preprocessing: scales to [-1, 1]) ---
    img_rgb = tf.image.resize(img, [224, 224])
    img_rgb = eff_preprocess(img_rgb)
    img_rgb = tf.ensure_shape(img_rgb, [224, 224, 3])

    return (srm_feat, img_rgb), tf.cast(label, tf.float32)

def augment_dual(inputs, label):
    srm, rgb = inputs
    srm = tf.image.random_flip_left_right(srm)
    rgb = tf.image.random_flip_left_right(rgb)
    srm = tf.image.random_brightness(srm, max_delta=0.05)
    srm = tf.clip_by_value(srm, 0.0, 1.0)
    return (srm, rgb), label

def build_dual_dataset(paths, labels, batch_size=TWOSTREAM_BATCH, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(min(len(paths), 10000), reshuffle_each_iteration=True)
    ds = ds.map(process_dual_stream, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment_dual, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

print("📦 Building Two-Stream datasets...")
train_dual_ds = build_dual_dataset(train_image_paths,      train_labels,  training=True)
val_dual_ds   = build_dual_dataset(validation_image_paths, val_labels,    training=False)
test_dual_ds  = build_dual_dataset(test_image_paths,       test_labels,   training=False)

# Sanity check
for (s, r), y in train_dual_ds.take(1):
    print("SRM batch shape:", s.shape)
    print("RGB batch shape:", r.shape)
    print("Label shape:    ", y.shape)

print("✅ Two-Stream datasets ready")

In [ ]:
# Cell 44
# =================== TWO-STREAM MODEL ARCHITECTURE ===================

import tensorflow as tf
from tensorflow.keras import layers, Model

# ── load best single-stream models from disk ─────────────────────────────────
encoder_path     = find_latest_model("final_encoder_24ch.keras")
eff_stage2_path  = stage2_path  # already defined by cell 38

srm_encoder  = tf.keras.models.load_model(encoder_path,    compile=False)
eff_backbone = tf.keras.models.load_model(eff_stage2_path, compile=False)

# Extract the inner EfficientNetB0 backbone (before the classification head)
inner_eff = eff_backbone.get_layer("efficientnetb0")

# Freeze both backbones for Phase 1 (train only the fusion head)
srm_encoder.trainable = False
inner_eff.trainable   = False

L2 = tf.keras.regularizers.l2(1e-4)

# ── Stream 1: SRM / noise ────────────────────────────────────────────────────
srm_input  = layers.Input(shape=(256, 256, 24), name="srm_input")
srm_latent = srm_encoder(srm_input, training=False)              # (B, 16, 16, 256)
srm_vec    = layers.GlobalAveragePooling2D(name="srm_gap")(srm_latent)
srm_vec    = layers.Dense(256, activation="relu",
                           kernel_regularizer=L2, name="srm_proj")(srm_vec)
srm_vec    = layers.Dropout(0.3, name="srm_drop")(srm_vec)

# ── Stream 2: RGB / appearance ───────────────────────────────────────────────
rgb_input  = layers.Input(shape=(224, 224, 3), name="rgb_input")
rgb_feat   = inner_eff(rgb_input, training=False)                # (B, 7, 7, 1280)
rgb_vec    = layers.GlobalAveragePooling2D(name="rgb_gap")(rgb_feat)
rgb_vec    = layers.Dense(256, activation="relu",
                           kernel_regularizer=L2, name="rgb_proj")(rgb_vec)
rgb_vec    = layers.Dropout(0.3, name="rgb_drop")(rgb_vec)

# ── Fusion head ──────────────────────────────────────────────────────────────
fused = layers.Concatenate(name="fusion")([srm_vec, rgb_vec])   # (B, 512)

x = layers.Dense(256, kernel_regularizer=L2, name="fuse_dense1")(fused)
x = layers.BatchNormalization(name="fuse_bn1")(x)
x = layers.Activation("relu", name="fuse_act1")(x)
x = layers.Dropout(0.5, name="fuse_drop1")(x)

x = layers.Dense(128, kernel_regularizer=L2, name="fuse_dense2")(x)
x = layers.BatchNormalization(name="fuse_bn2")(x)
x = layers.Activation("relu", name="fuse_act2")(x)
x = layers.Dropout(0.4, name="fuse_drop2")(x)

output = layers.Dense(1, activation="sigmoid", name="prob_fake")(x)

two_stream_model = Model(
    inputs=[srm_input, rgb_input],
    outputs=output,
    name="TwoStream_Deepfake"
)

two_stream_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=["accuracy",
             tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

two_stream_model.summary(line_length=100)
print("✅ Two-Stream model built")

In [ ]:
# Cell 45
# =================== TWO-STREAM TRAINING — PHASE 1 (frozen backbones) ===================

TS_SAVE_DIR    = TS_DIR
ts_phase1_path = os.path.join(TS_DIR, "two_stream_phase1_best.keras")
ts_final_path  = os.path.join(TS_DIR, "two_stream_final.keras")

_existing_ts = find_latest_model("two_stream_final.keras") if SKIP_IF_SAVED else None
if _existing_ts:
    print(f"⏭️  SKIP_IF_SAVED=True — loading Two-Stream from:\n   {_existing_ts}")
    two_stream_model = tf.keras.models.load_model(_existing_ts, compile=False)
    ts_final_path = _existing_ts
    print("✅ Two-Stream model loaded — skipping Phase 1 & 2 training")
else:
    callbacks_ts1 = [
        tf.keras.callbacks.ModelCheckpoint(ts_phase1_path, monitor="val_auc", mode="max",
                                            save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5,
                                          restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5,
                                              patience=2, min_lr=1e-6, verbose=1),
    ]
    print("🚀 Two-Stream Phase 1 — fusion head only (backbones frozen)...")
    history_ts1 = two_stream_model.fit(
        train_dual_ds, validation_data=val_dual_ds,
        epochs=15, class_weight=srm_class_weights,
        callbacks=callbacks_ts1, verbose=1
    )
    print("✅ Phase 1 complete")

In [ ]:
# Cell 46
# =================== TWO-STREAM TRAINING — PHASE 2 (partial unfreeze) ===================

if _existing_ts:
    print("⏭️  Phase 2 skipped — model was loaded from saved checkpoint")
else:
    two_stream_model = tf.keras.models.load_model(ts_phase1_path, compile=False)

    srm_enc = two_stream_model.get_layer("srm_encoder_24ch")
    srm_enc.trainable = True
    for layer in srm_enc.layers[:int(len(srm_enc.layers) * 0.70)]:
        layer.trainable = False

    eff_b = two_stream_model.get_layer("efficientnetb0")
    eff_b.trainable = True
    for layer in eff_b.layers[:int(len(eff_b.layers) * 0.70)]:
        layer.trainable = False

    trainable = sum(tf.size(v).numpy() for v in two_stream_model.trainable_variables)
    print(f"Trainable params after unfreeze: {trainable:,}")

    two_stream_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5, clipnorm=1.0),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
                 tf.keras.metrics.Precision(name="precision"),
                 tf.keras.metrics.Recall(name="recall")]
    )

    ts_phase2_path = os.path.join(TS_DIR, "two_stream_phase2_best.keras")
    callbacks_ts2 = [
        tf.keras.callbacks.ModelCheckpoint(ts_phase2_path, monitor="val_auc", mode="max",
                                            save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6,
                                          restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5,
                                              patience=3, min_lr=1e-7, verbose=1),
    ]

    print("🚀 Two-Stream Phase 2 — end-to-end fine-tuning...")
    history_ts2 = two_stream_model.fit(
        train_dual_ds, validation_data=val_dual_ds,
        epochs=20, class_weight=srm_class_weights,
        callbacks=callbacks_ts2, verbose=1
    )
    print("✅ Phase 2 complete")
    save_model_versioned(two_stream_model, TS_DIR, "two_stream_final")

In [ ]:
# Cell 47
# =================== TWO-STREAM EVALUATION ===================

import numpy as np
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

best_ts = tf.keras.models.load_model(ts_phase2_path, compile=False)

y_true_ts, y_prob_ts = [], []
for (s_batch, r_batch), y_batch in test_dual_ds:
    preds = best_ts.predict([s_batch, r_batch], verbose=0).flatten()
    y_true_ts.extend(y_batch.numpy())
    y_prob_ts.extend(preds)

y_true_ts = np.array(y_true_ts).astype(int)
y_prob_ts = np.array(y_prob_ts)
y_pred_ts = (y_prob_ts > 0.5).astype(int)

print("📊 Two-Stream Classification Report:")
print(classification_report(y_true_ts, y_pred_ts, target_names=["REAL", "FAKE"], digits=4))
print(f"Accuracy : {accuracy_score(y_true_ts, y_pred_ts):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_true_ts, y_prob_ts):.4f}")

# Ensemble Model
Combines all three classifiers:
1. **CNN-SRM** — noise/frequency domain stream  
2. **EfficientNetB0** — RGB appearance stream  
3. **Two-Stream** — joint noise + appearance model  

We first try simple **uniform averaging**, then find the **optimal per-model weight** on the validation set using Nelder-Mead optimisation.

In [ ]:
# Cell 49
# =================== COLLECT PREDICTIONS FROM ALL THREE MODELS ===================
# Run once on VALIDATION (for weight search) and TEST (for final metrics).

import numpy as np

def predict_srm(model, dataset):
    yt, yp = [], []
    for x, y in dataset:
        yt.extend(y.numpy())
        yp.extend(model.predict(x, verbose=0).flatten())
    return np.array(yt).astype(int), np.array(yp)

def predict_rgb(model, dataset):
    yt, yp = [], []
    for x, y in dataset:
        yt.extend(y.numpy())
        yp.extend(model.predict(x, verbose=0).flatten())
    return np.array(yt).astype(int), np.array(yp)

def predict_dual(model, dataset):
    yt, yp = [], []
    for (s, r), y in dataset:
        yt.extend(y.numpy())
        yp.extend(model.predict([s, r], verbose=0).flatten())
    return np.array(yt).astype(int), np.array(yp)

# Load models (already in memory or reload from disk)
srm_model_path = find_latest_model("final_cnn_srm.keras")
eff_model_path = stage2_path   # defined by cell 38
ts_model_path  = ts_final_path # defined by cell 45

try:
    model_srm = tf.keras.models.load_model(srm_model_path, compile=False)
    model_eff = tf.keras.models.load_model(eff_model_path, compile=False)
    model_ts  = tf.keras.models.load_model(ts_model_path,  compile=False)
    print("✅ All three models loaded for ensemble")
except Exception as e:
    print(f"❌ Could not load models: {e}")
    raise

print("🔮 Running inference — validation set...")
val_y,  val_p_srm = predict_srm(model_srm, val_srm_dataset)
_,      val_p_eff = predict_rgb(model_eff, val_rgb_ds)
_,      val_p_ts  = predict_dual(model_ts, val_dual_ds)

print("🔮 Running inference — test set...")
test_y,  test_p_srm = predict_srm(model_srm, test_srm_dataset)
_,       test_p_eff = predict_rgb(model_eff, test_rgb_ds)
_,       test_p_ts  = predict_dual(model_ts, test_dual_ds)

print("✅ Predictions collected")
print(f"  Val  samples: {len(val_y):,} | Test samples: {len(test_y):,}")

In [ ]:
# Cell 50
# =================== ENSEMBLE — UNIFORM AVERAGE ===================

import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

def evaluate_ensemble(y_true, p_srm, p_eff, p_ts, w=(1/3, 1/3, 1/3), threshold=0.5, name=""):
    w1, w2, w3 = w
    p_ens = w1 * p_srm + w2 * p_eff + w3 * p_ts
    y_pred = (p_ens > threshold).astype(int)
    acc  = accuracy_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, p_ens)
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"  Weights → SRM: {w1:.3f}  EfficientNet: {w2:.3f}  TwoStream: {w3:.3f}")
    print(f"  Accuracy : {acc:.4f}   ROC-AUC: {auc:.4f}")
    print(f"{'='*55}")
    print(classification_report(y_true, y_pred, target_names=["REAL", "FAKE"], digits=4))
    return p_ens

# --- Uniform ensemble (equal weights) ---
print("📊 Uniform Ensemble (1/3 each) — TEST SET")
ens_uniform = evaluate_ensemble(
    test_y, test_p_srm, test_p_eff, test_p_ts,
    w=(1/3, 1/3, 1/3), name="Uniform Ensemble"
)

In [ ]:
# Cell 51
# =================== ENSEMBLE — LEARNED WEIGHTS (optimised on validation) ===================
# Uses Nelder-Mead minimisation to find the weights that maximise val-set AUC.

import numpy as np
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score, accuracy_score

def neg_auc(raw_weights):
    """Objective: negative AUC on validation set (we minimise → maximise AUC)."""
    w = np.array(raw_weights)
    w = np.abs(w) / np.sum(np.abs(w))          # normalise to sum = 1
    p = w[0]*val_p_srm + w[1]*val_p_eff + w[2]*val_p_ts
    return -roc_auc_score(val_y, p)

result = minimize(
    neg_auc,
    x0=[1/3, 1/3, 1/3],
    method="Nelder-Mead",
    options={"maxiter": 500, "xatol": 1e-5, "fatol": 1e-6}
)

raw_w = np.abs(result.x)
opt_w = raw_w / raw_w.sum()
print(f"Optimal weights found  →  SRM: {opt_w[0]:.3f}  |  EfficientNet: {opt_w[1]:.3f}  |  TwoStream: {opt_w[2]:.3f}")
print(f"Validation AUC with optimal weights: {-result.fun:.4f}")

# --- Evaluate on TEST set with optimal weights ---
print("\n📊 Optimised Ensemble — TEST SET")
ens_opt = evaluate_ensemble(
    test_y, test_p_srm, test_p_eff, test_p_ts,
    w=tuple(opt_w), name="Optimised Ensemble"
)

# --- Summary table ---
import pandas as pd

rows = {
    "CNN-SRM":          (accuracy_score(test_y, (test_p_srm > 0.5).astype(int)),
                         roc_auc_score(test_y, test_p_srm)),
    "EfficientNetB0":   (accuracy_score(test_y, (test_p_eff > 0.5).astype(int)),
                         roc_auc_score(test_y, test_p_eff)),
    "Two-Stream":       (accuracy_score(test_y, (test_p_ts  > 0.5).astype(int)),
                         roc_auc_score(test_y, test_p_ts)),
    "Ensemble (uniform)":(accuracy_score(test_y, (ens_uniform > 0.5).astype(int)),
                          roc_auc_score(test_y, ens_uniform)),
    "Ensemble (optimal)":(accuracy_score(test_y, (ens_opt    > 0.5).astype(int)),
                          roc_auc_score(test_y, ens_opt)),
}

df = pd.DataFrame(rows, index=["Accuracy", "ROC-AUC"]).T
print("\n" + "="*50)
print("MODEL COMPARISON — TEST SET")
print("="*50)
print(df.to_string(float_format="{:.4f}".format))
TARGET_MET = df["Accuracy"].max() >= 0.85
print(f"\n🎯 Target (≥0.85 accuracy) {'ACHIEVED ✅' if TARGET_MET else 'NOT YET reached — continue fine-tuning'}")